# Addressing Advisor Feedback — Window Clarification, Band Redefinition, Pre/Post-Poke EDA

**This notebook directly addresses two pieces of feedback:**

1. **"Not sure what window you used since you imported some functions from other script. Note that at
   2000ms is when the animal POKE IN, not 0ms."** — The windowing logic in earlier notebooks lived
   inside `src/preprocessing.py` and was called as a single function, which made it hard to verify from
   the notebook alone what was actually happening. This notebook makes every step of the windowing
   fully explicit and inline, with printed, real numbers (not assumed conventions) proving exactly where
   "0ms" is anchored. **To be precise about our reference point: in this notebook, "0ms" always means
   the real elapsed time (read directly from each session's own `TimeBin` channel) at the sample where
   the trial marker fires. We verified in notebook 01's audit that this moment is the odor onset / poke
   in event itself (0 sample offset between the trial marker and the odor channel turning on), not some
   other epoch convention with a separate pre-period. We are not assuming a fixed epoch structure from
   elsewhere, we're reading the real timestamp directly out of the data for every trial.**

2. **"Usually 4-12 Hz are all called theta. You should also include high gamma. Some EDA about how
   frequency distributions differ before and after poke-in, and within-class variation, would be
   interesting."** — Addressed in Sections 3-5: redefined theta as 4-12 Hz (previously split into a
   narrower 4-8 Hz theta and separate 8-12 Hz alpha), added a high gamma band, and added EDA comparing
   band power before vs. after poke-in, and the spread of band power within each class.

Everything here uses Mitt, consistent with prior notebooks. Presentation deadline: Sep 1-2.


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing import build_labels, get_sampling_rate


## 1. Load raw data

In [ ]:
session_dir = '../data/raw/080718_mitt'
session_name = '080718_mitt'

bvr = np.load(f'{session_dir}/{session_name}_bvr.npz', allow_pickle=True)
bvr_data = bvr['data']
bvr_keys = bvr['keys'].tolist()

lfp = np.load(f'{session_dir}/{session_name}_lfp.npz', allow_pickle=True)
lfp_data = lfp['data']
lfp_keys = lfp['keys'].tolist()

fs = get_sampling_rate(bvr_data, bvr_keys)
labels = build_labels(bvr_data, bvr_keys)
timebin = bvr_data[bvr_keys.index('TimeBin')]

print(f"Sampling rate: {fs:.2f} Hz")
print(f"Trials found: {len(labels['trial_idx'])}")


## 2. Make the windowing fully explicit (addressing feedback point 1)

For the very first trial, we print out, in real units, exactly what "0ms" refers to, and exactly what
our window covers. No hidden function calls, every number here is computed inline and printed so it can
be checked directly.


In [ ]:
# window lengths, defined explicitly here rather than buried in a config or imported function
PRE_MS = 500    # baseline window BEFORE the trial marker
POST_MS = 500   # window AFTER the trial marker (matches what earlier notebooks used)

pre_samples = int(round(PRE_MS / 1000 * fs))
post_samples = int(round(POST_MS / 1000 * fs))

example_trial_sample_idx = labels['trial_idx'][0]
example_trial_real_time_sec = timebin[example_trial_sample_idx]

print("=== Explicit windowing check, trial 0 ===")
print(f"Trial marker fires at SAMPLE INDEX: {example_trial_sample_idx}")
print(f"Trial marker fires at REAL ELAPSED TIME (from TimeBin): {example_trial_real_time_sec:.4f} sec")
print(f"This is what we call t=0ms for this trial, confirmed odor-onset/poke-in, NOT an assumed")
print(f"epoch convention.")
print()
print(f"PRE-poke window: samples [{example_trial_sample_idx - pre_samples}, {example_trial_sample_idx}]")
print(f"  -> real time [{timebin[example_trial_sample_idx - pre_samples]:.4f}, {example_trial_real_time_sec:.4f}] sec")
print(f"  -> relative to poke-in: [-{PRE_MS}ms, 0ms]")
print()
print(f"POST-poke window: samples [{example_trial_sample_idx}, {example_trial_sample_idx + post_samples}]")
print(f"  -> real time [{example_trial_real_time_sec:.4f}, {timebin[example_trial_sample_idx + post_samples]:.4f}] sec")
print(f"  -> relative to poke-in: [0ms, +{POST_MS}ms]")


## 3. Extract PRE and POST windows for every trial

Same explicit approach, applied to all trials. Trials too close to the start or end of the recording
(where a full pre or post window wouldn't fit) are dropped and reported.


In [ ]:
def extract_window(lfp_data, center_idx, n_samples_before, n_samples_after):
    """Explicit, inline windowing: [center_idx - before, center_idx + after)."""
    start = center_idx - n_samples_before
    end = center_idx + n_samples_after
    return lfp_data[:, start:end], start, end

n_timepoints = lfp_data.shape[1]
pre_windows, post_windows, kept_idx = [], [], []

for t in labels['trial_idx']:
    if t - pre_samples < 0 or t + post_samples > n_timepoints:
        continue  # too close to recording start/end, drop
    pre_w, _, _ = extract_window(lfp_data, t, pre_samples, 0)
    post_w, _, _ = extract_window(lfp_data, t, 0, post_samples)
    pre_windows.append(pre_w)
    post_windows.append(post_w)
    kept_idx.append(t)

pre_windows = np.stack(pre_windows, axis=0)   # (n_trials, n_channels, pre_samples)
post_windows = np.stack(post_windows, axis=0) # (n_trials, n_channels, post_samples)
kept_idx = np.array(kept_idx)

kept_mask = np.isin(labels['trial_idx'], kept_idx)
inseq_outseq = labels['inseq_outseq'][kept_mask]
odor_id = labels['odor_id'][kept_mask]

print("pre_windows shape:", pre_windows.shape)
print("post_windows shape:", post_windows.shape)
print("Trials kept:", len(kept_idx), "of", len(labels['trial_idx']))


## 4. Redefine frequency bands (addressing feedback point 2)

Per feedback: theta widened to 4-12 Hz (previously split into separate 4-8 Hz theta and 8-12 Hz alpha
bands, now merged, matching the more common convention), and a high gamma band added. Band boundaries
below are a standard reference set, adjust further if the lab uses different exact cutoffs.

| Band | Range (Hz) | Change from earlier notebook |
|---|---|---|
| Delta | 1-4 | unchanged |
| Theta | 4-12 | widened, previously 4-8 (theta) + 8-12 (alpha) as two separate bands |
| Beta | 12-30 | unchanged |
| Low gamma | 30-80 | unchanged |
| High gamma | 80-150 | **new**, added per feedback |


In [ ]:
BANDS = {
    'delta': (1, 4),
    'theta': (4, 12),
    'beta': (12, 30),
    'low_gamma': (30, 80),
    'high_gamma': (80, 150),
}

def band_power(window_batch, fs, bands):
    """window_batch: (n_trials, n_channels, n_samples) -> (n_trials, n_channels * n_bands)"""
    n_trials, n_channels, n_samples = window_batch.shape
    freqs = np.fft.rfftfreq(n_samples, d=1 / fs)
    band_masks = {name: (freqs >= lo) & (freqs < hi) for name, (lo, hi) in bands.items()}

    n_features = n_channels * len(bands)
    X = np.zeros((n_trials, n_features))
    feature_names = []
    col = 0
    for ch in range(n_channels):
        fft_vals = np.fft.rfft(window_batch[:, ch, :], axis=1)
        power = np.abs(fft_vals) ** 2
        for band_name, mask in band_masks.items():
            X[:, col] = power[:, mask].sum(axis=1)
            feature_names.append(f"ch{ch}_{band_name}")
            col += 1
    return X, feature_names

X_pre, feature_names = band_power(pre_windows, fs, BANDS)
X_post, _ = band_power(post_windows, fs, BANDS)

print("Feature matrix shape (per window):", X_post.shape, "->", len(BANDS), "bands x", pre_windows.shape[1], "channels")


## 5. EDA: band power before vs. after poke-in

Averaged across all channels and all trials, per band. This directly answers the feedback's question:
does frequency content actually change around poke-in, and if so, which bands show it?


In [ ]:
band_names = list(BANDS.keys())
n_channels = pre_windows.shape[1]

# reshape (n_trials, n_channels * n_bands) -> mean power per band, averaged over trials and channels
def mean_power_per_band(X, band_names, n_channels):
    X_reshaped = X.reshape(X.shape[0], n_channels, len(band_names))  # (trials, channels, bands)
    return X_reshaped.mean(axis=(0, 1))  # average over trials and channels -> (bands,)

pre_band_means = mean_power_per_band(X_pre, band_names, n_channels)
post_band_means = mean_power_per_band(X_post, band_names, n_channels)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(band_names))
width = 0.35
ax.bar(x - width/2, np.log1p(pre_band_means), width, label='Pre-poke (-500 to 0ms)')
ax.bar(x + width/2, np.log1p(post_band_means), width, label='Post-poke (0 to +500ms)')
ax.set_xticks(x)
ax.set_xticklabels(band_names)
ax.set_ylabel('log(1 + mean power)')
ax.set_title('Band power before vs. after poke-in (Mitt, all channels/trials averaged)')
ax.legend()
plt.tight_layout()
plt.show()

print("Pre-poke mean power by band:", dict(zip(band_names, np.round(pre_band_means, 2))))
print("Post-poke mean power by band:", dict(zip(band_names, np.round(post_band_means, 2))))


## 6. EDA: within-class variation

How much does band power vary WITHIN a class (e.g. across all InSeq trials, or across all trials of one
odor), versus between classes? A feature that's wildly variable within its own class is less useful for
classification even if its average differs between classes. We look at theta power (post-poke), the
band the feedback specifically called out, averaged across channels per trial, split by class.


In [ ]:
theta_idx = band_names.index('theta')
# average theta power across all channels for each trial (post-poke window)
X_post_reshaped = X_post.reshape(X_post.shape[0], n_channels, len(band_names))
theta_power_per_trial = np.log1p(X_post_reshaped[:, :, theta_idx].mean(axis=1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# InSeq vs OutSeq
data_by_inseq = [theta_power_per_trial[inseq_outseq == 0], theta_power_per_trial[inseq_outseq == 1]]
axes[0].boxplot(data_by_inseq, labels=['OutSeq', 'InSeq'])
axes[0].set_ylabel('log(1 + theta power), averaged across channels')
axes[0].set_title('Theta power by InSeq/OutSeq')

# by odor, InSeq trials only
inseq_mask = (inseq_outseq == 1)
odor_labels = ['A', 'B', 'C', 'D', 'E']
data_by_odor = [theta_power_per_trial[inseq_mask][odor_id[inseq_mask] == i] for i in range(5)]
axes[1].boxplot(data_by_odor, labels=odor_labels)
axes[1].set_ylabel('log(1 + theta power), averaged across channels')
axes[1].set_title('Theta power by odor identity (InSeq trials only)')

plt.tight_layout()
plt.show()


### What to look for in the boxplots above

If the boxes for different classes barely overlap, that band/class combination likely carries real
discriminative signal. If the boxes overlap heavily (as raw theta power often will, biological signals
are noisy), that tells us this single averaged feature alone isn't enough, consistent with why we're
using many features (band x channel combinations) and a trained classifier rather than a simple
threshold on one number.


## 7. Text-only results summary

Same pattern as the earlier notebooks: everything above printed as plain, readable JSON, and saved to
`outputs/logs/`, so results can be shared and interpreted without screenshots.


In [ ]:
import json as _json
import os

results_summary = {
    "session": session_name,
    "purpose": "Addressing advisor feedback: explicit windowing, redefined bands, pre/post-poke EDA",
    "windowing": {
        "reference_point": "0ms = real TimeBin value at trial marker (confirmed = poke-in/odor onset per notebook 01 audit)",
        "pre_window_ms": PRE_MS,
        "post_window_ms": POST_MS,
        "example_trial_sample_idx": int(example_trial_sample_idx),
        "example_trial_real_time_sec": round(float(example_trial_real_time_sec), 4),
    },
    "bands": {name: list(rng) for name, rng in BANDS.items()},
    "n_trials_kept": int(len(kept_idx)),
    "n_trials_dropped": int(len(labels['trial_idx']) - len(kept_idx)),
    "band_power_pre_vs_post": {
        "note": "mean power per band, averaged across all channels and trials, log1p NOT applied here (raw means)",
        "pre_poke": {name: round(float(v), 4) for name, v in zip(band_names, pre_band_means)},
        "post_poke": {name: round(float(v), 4) for name, v in zip(band_names, post_band_means)},
    },
    "theta_power_by_inseq_outseq": {
        "note": "log1p(theta power), averaged across channels, per trial -- summary stats by class",
        "OutSeq": {
            "n": int(len(data_by_inseq[0])),
            "mean": round(float(np.mean(data_by_inseq[0])), 4),
            "std": round(float(np.std(data_by_inseq[0])), 4),
            "min": round(float(np.min(data_by_inseq[0])), 4),
            "max": round(float(np.max(data_by_inseq[0])), 4),
        },
        "InSeq": {
            "n": int(len(data_by_inseq[1])),
            "mean": round(float(np.mean(data_by_inseq[1])), 4),
            "std": round(float(np.std(data_by_inseq[1])), 4),
            "min": round(float(np.min(data_by_inseq[1])), 4),
            "max": round(float(np.max(data_by_inseq[1])), 4),
        },
    },
    "theta_power_by_odor": {
        odor_labels[i]: {
            "n": int(len(data_by_odor[i])),
            "mean": round(float(np.mean(data_by_odor[i])), 4),
            "std": round(float(np.std(data_by_odor[i])), 4),
        }
        for i in range(5)
    },
}

print(_json.dumps(results_summary, indent=2))

os.makedirs('../outputs/logs', exist_ok=True)
with open('../outputs/logs/band_redefinition_eda_results.json', 'w') as f:
    _json.dump(results_summary, f, indent=2)
print("\nSaved to outputs/logs/band_redefinition_eda_results.json")


## Summary

- **Windowing is now fully explicit and verifiable**: every window's start/end is computed and printed
  in real elapsed time (from `TimeBin`), anchored to the trial marker, which notebook 01's audit
  confirmed is the true poke-in/odor-onset moment, not an assumed convention.
- **Theta redefined to 4-12 Hz** (previously split into 4-8 Hz theta + 8-12 Hz alpha), **high gamma
  (80-150 Hz) added**.
- **Pre/post-poke and within-class EDA added**, both as explicit visualizations, ready to bring to the
  Sep 1-2 presentation.

**Next, per the plan:** re-run the classification models (GLM, and eventually RNN) using this updated
band definition and the POST-poke window, to see whether the redefined bands change performance versus
notebook 05's original results (InSeq/OutSeq 59.2%, odor 30.5%). That's the natural following step, now
that the feedback on windowing and band definitions has been addressed.
